# Independent review of the bounded semantic follow-up

Purpose: independently reconcile completed Notebook04 evidence, original wall-clock cap, frozen sources, all paired training trajectories, and every fixed final epoch-20 probe prediction. This notebook performs CPU reads and arithmetic only; it never trains models or evaluates the official test set.

Inputs: completed `semantic-followup-001` plan, protocols, checkpoints, feature caches, predictions, and reports. Expected runtime: 1–3 minutes on CPU. Outputs: `artifacts/notebook_04/<RUN_ID>/independent_review.json`, an inspectable audit result, the final report, and embedded PNG figures. Existing experiment evidence is not rewritten.

Checkpoint location: original training and probe evidence remains under `checkpoints/<RUN_ID>/`; this review needs no experiment checkpoint. Resume: rerun this notebook top-to-bottom with the identical RUN_ID. A missing/incomplete experiment raises instead of approving a partial comparison. Tagged cells contain reusable audit definitions; the final execution/display cells are deliberately untagged so imports cannot trigger a review.

The audit verifies numerical evidence. Visual inspection of the displayed PNGs is a separate step and is not asserted by numerical PASS.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa")
_boot = {"__file__": str(ROOT / "fmm_jepa/bootstrap.ipynb")}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)


In [ ]:
import copy, hashlib, importlib.metadata, itertools, json, math, time
from pathlib import Path
import numpy as np
import torch
from torch.nn import functional as F
from fmm_jepa.training import definition_hashes, digest_file, atomic_json
from fmm_jepa.food101_training import validate_training_manifest
from fmm_jepa.food101_evaluation import encoder_state_hash, validate_feature_cache, _load_probe_checkpoint

def _review_json(path):
    return json.loads(Path(path).read_text())

def _review_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def _record_verified(record):
    path = Path(record["path"])
    assert digest_file(path) == record["sha256"], "Evidence file hash changed: " + str(path)
    return path

def _finite_tensor_tree(value, label="root"):
    if isinstance(value, torch.Tensor):
        if value.is_floating_point() or value.is_complex():
            assert torch.isfinite(value).all(), "Nonfinite tensor: " + label
    elif isinstance(value, dict):
        for key, child in value.items(): _finite_tensor_tree(child, label + "." + str(key))
    elif isinstance(value, (tuple, list)):
        for index, child in enumerate(value): _finite_tensor_tree(child, label + "." + str(index))

def _review_source_hashes(root):
    root = Path(root)
    hashes = definition_hashes(root)
    for path in sorted((root / "experiments/04_followup").glob("*.ipynb")):
        notebook = _review_json(path)
        code = ["".join(cell["source"]) for cell in notebook["cells"]
                if cell["cell_type"] == "code" and "ahr-export" in cell.get("metadata", {}).get("tags", [])]
        hashes[str(path.relative_to(root))] = hashlib.sha256("\n".join(code).encode()).hexdigest()
    return hashes

def _read_review_caches(feature_folder, train_ids, dev_ids, heldout_ids, summary, selected):
    caches, evidence = {}, {}
    for split, ids in (("train", list(train_ids)), ("development", list(dev_ids))):
        found = []
        for path in Path(feature_folder).glob("*.json"):
            metadata = _review_json(path)
            if metadata.get("identity", {}).get("sample_ids") == ids:
                found.append((path, metadata))
        assert len(found) == 1, "Expected one complete cache per split: " + split
        meta_path, metadata = found[0]
        path = meta_path.with_suffix(".pt")
        assert digest_file(path) == metadata["sha256"]
        cache = torch.load(path, map_location="cpu", weights_only=True)
        validate_feature_cache(cache, metadata["identity"])
        assert cache["sample_ids"] == ids and not set(ids) & set(heldout_ids)
        assert cache["identity"]["checkpoint_hash"] == summary["checkpoint_hash"]
        identity_key, content_key = (("train_identity", "train_cache") if split == "train" else ("dev_identity", "dev_cache"))
        assert selected["identity"][identity_key] == cache["identity"]
        assert selected["identity"][content_key] == cache["content_hash"]
        caches[split] = cache
        evidence[split] = {"path": str(path), "sha256": metadata["sha256"], "examples": len(ids)}
    assert not set(caches["train"]["sample_ids"]) & set(caches["development"]["sample_ids"])
    return caches, evidence

def _reconstruct_probe_predictions(selected, caches, predictions, summary):
    """Independent CPU affine logits, normalization, all top1/top5 predictions."""
    assert selected["encoder_checkpoint_hash"] == summary["checkpoint_hash"]
    assert selected["probe_hash"] == summary["probe_hash"]
    state = selected["state_dict"]
    assert set(state) == {"weight", "bias"}
    assert encoder_state_hash(state) == summary["probe_hash"]
    train = caches["train"]["features"].float()
    dev = caches["development"]["features"].float()
    labels = caches["development"]["labels"]
    stats = selected["standardization"]
    mean = train.mean(0) if stats["enabled"] else torch.zeros(train.shape[1])
    std = train.std(0, unbiased=False).clamp_min(1e-6) if stats["enabled"] else torch.ones(train.shape[1])
    torch.testing.assert_close(stats["mean"], mean, rtol=1e-6, atol=1e-7)
    torch.testing.assert_close(stats["std"], std, rtol=1e-6, atol=1e-7)
    assert stats["fit_examples"] == len(train)
    assert state["weight"].shape == (101, train.shape[1]) and state["bias"].shape == (101,)
    _finite_tensor_tree(state, "classifier")
    normalized = (dev - stats["mean"]) / stats["std"]
    with torch.inference_mode():
        logits = torch.cat([F.linear(part, state["weight"], state["bias"])
                            for part in normalized.split(int(selected["config"]["batch_size"]))])
    assert torch.isfinite(logits).all()
    top = logits.topk(5, dim=1).indices
    assert len(predictions) == len(dev)
    assert [row["sample_id"] for row in predictions] == caches["development"]["sample_ids"]
    for index, row in enumerate(predictions):
        assert row["model_id"] == summary["model_id"] and row["split"] == "development"
        assert row["checkpoint_hash"] == summary["checkpoint_hash"] and row["probe_hash"] == summary["probe_hash"]
        assert row["label"] == int(labels[index])
        assert row["top1_prediction"] == int(top[index, 0])
        assert row["top5_predictions"] == top[index].tolist()
        assert row["correct_top1"] == int(top[index, 0] == labels[index])
        assert row["correct_top5"] == int((top[index] == labels[index]).any())
    metrics = {"top1": float((top[:, 0] == labels).double().mean()),
               "top5": float((top == labels[:, None]).any(1).double().mean()),
               "loss": float(F.cross_entropy(logits, labels))}
    for key in ("top1", "top5"):
        assert np.isclose(metrics[key], summary["metrics"][key], rtol=0, atol=1e-7)
    assert np.isclose(metrics["loss"], summary["metrics"]["loss"], rtol=1e-6, atol=1e-7)
    return metrics

def reconcile_final_epoch_probe(root, artifact, run_id, model_id, saved, split):
    summary = saved["probe"]["summary"]
    assert summary["selection"] == "fixed_final_epoch" and summary["selected_epoch"] == 20
    assert summary["completed_epochs"] == 20 and summary["successful_probe_updates"] == 800
    assert summary["historical_best_dev_artifact_used"] is False
    assert summary["budget"]["selection"] == "fixed_final_epoch"
    assert summary["train_examples"] == 10100 and summary["train_classes"] == 101
    assert summary["encoder_unchanged"] and summary["standardization_fit_scope"] == "probe_train_only"
    path = Path(summary["selected_probe_path"])
    assert path.name == "final_epoch_probe.pt"
    selected = torch.load(path, map_location="cpu", weights_only=True)
    assert selected["selection"] == "fixed_final_epoch" and selected["selected_epoch"] == 20
    checkpoint_dir = Path(root) / "checkpoints" / run_id / "probes" / model_id
    assert path.resolve() == (checkpoint_dir / "final_epoch_probe.pt").resolve()
    manifest = _review_json(checkpoint_dir / "probe_manifest.json")
    assert manifest["resume_notebook"] == "notebooks/04_Bounded_Semantic_Followup.ipynb"
    assert manifest["resume_cell"] == "launch-bounded-followup"
    latest = _load_probe_checkpoint(checkpoint_dir, selected["identity"])
    assert latest["completed_epochs"] == 20 and latest["active_order"] is None and latest["next_offset"] == 0
    assert latest["successful_updates"] == latest["attempted_updates"] == 800
    assert latest["initialization_hash"] == summary["initialization_hash"]
    assert encoder_state_hash(latest["classifier"]) == summary["probe_hash"]
    for name in ("mean", "std"):
        assert torch.equal(latest["standardization"][name], selected["standardization"][name])
    _finite_tensor_tree(latest["classifier"], "probe.classifier")
    _finite_tensor_tree(latest["optimizer"], "probe.optimizer")
    assert len(manifest["retained"]) >= 2 and manifest.get("best")
    retained_files = {entry["file"] for entry in manifest["retained"]}
    assert manifest["best"]["file"] in retained_files and manifest["latest"]["file"] in retained_files
    for entry in manifest["retained"]:
        if entry["file"] == manifest["latest"]["file"]: continue
        retained = torch.load(checkpoint_dir / entry["file"], map_location="cpu", weights_only=True)
        _finite_tensor_tree(retained["classifier"], "retained_probe.classifier")
        _finite_tensor_tree(retained["optimizer"], "retained_probe.optimizer")
        del retained
    caches, evidence = _read_review_caches(Path(artifact) / "features" / model_id,
        split["training"], split["development"], split["heldout"], summary, selected)
    for name, record in saved["feature_caches"].items():
        assert _record_verified(record).resolve() == Path(evidence[name]["path"]).resolve()
    metrics = _reconstruct_probe_predictions(selected, caches, saved["probe"]["predictions"], summary)
    recorded = [row for row in latest["metrics"] if row["split"] == "development" and row["epoch"] == 20]
    assert len(recorded) == 1
    for key in metrics: assert np.isclose(recorded[0][key], metrics[key], rtol=1e-6, atol=1e-7)
    order_hash = _review_hash([(r["epoch"], r["split"], r["permutation_hash"]) for r in latest["metrics"]])
    return {"status": "PASS", "predictions_reconstructed": len(saved["probe"]["predictions"]),
        "metrics": metrics, "final_probe_sha256": digest_file(path), "feature_caches": evidence,
        "order_hash": order_hash, "normalization_recomputed_from_training_only": True,
        "authoritative_final_epoch": 20, "device": "cpu", "heldout_features_read": False}

def _independent_seed_contrasts(plan, evaluations, controls, reported):
    expected = []
    for arm in plan["arms"]:
        expected.append((arm["kind"] + ":" + arm["condition"] + " trained minus initialization", arm["seed"], arm["id"], controls[arm["id"]]))
    for seed in sorted({arm["seed"] for arm in plan["arms"]}):
        arms = [arm for arm in plan["arms"] if arm["seed"] == seed]
        for first, second in itertools.combinations(arms, 2):
            label = first["kind"] + ":" + first["condition"] + " minus " + second["kind"] + ":" + second["condition"]
            expected.append((label, seed, first["id"], second["id"]))
    rows = reported["per_seed"]
    assert len(rows) == len(expected)
    grouped = {}
    for (label, seed, first, second), row in zip(expected, rows):
        assert (row["contrast"], row["seed"], row["first"], row["second"]) == (label, seed, first, second)
        a = {x["sample_id"]: x for x in evaluations[first]["probe"]["predictions"]}
        b = {x["sample_id"]: x for x in evaluations[second]["probe"]["predictions"]}
        assert set(a) == set(b) and len(a) == 2020
        assert all(a[key]["label"] == b[key]["label"] for key in a)
        delta = 100 * sum(a[key]["correct_top1"] - b[key]["correct_top1"] for key in a) / len(a)
        first_score = 100 * sum(x["correct_top1"] for x in a.values()) / len(a)
        second_score = 100 * sum(x["correct_top1"] for x in b.values()) / len(b)
        for key, value in (("delta_percentage_points", delta), ("first_top1_percent", first_score), ("second_top1_percent", second_score)):
            assert math.isclose(row[key], value, abs_tol=1e-10), label + " " + key
        grouped.setdefault(label, []).append((seed, delta))
    assert set(grouped) == set(reported["across_seed"])
    for label, pairs in grouped.items():
        reported_group = reported["across_seed"][label]
        values = np.asarray([value for _, value in pairs], dtype=float)
        assert len(values) == reported_group["n_training_seeds"] == 3
        mean = float(sum(values) / len(values))
        sd = math.sqrt(sum((value - mean) ** 2 for value in values) / (len(values) - 1))
        half = math.sqrt(2 * .95**2 / (1 - .95**2)) * sd / math.sqrt(len(values))
        assert math.isclose(reported_group["mean_percentage_points"], mean, abs_tol=1e-10)
        assert math.isclose(reported_group["sd_percentage_points"], sd, abs_tol=1e-10)
        np.testing.assert_allclose(reported_group["approximate_t_ci95_percentage_points"], [mean - half, mean + half], rtol=0, atol=1e-8)
        np.testing.assert_allclose(reported_group["range_percentage_points"], [min(values), max(values)], rtol=0, atol=1e-10)
        assert reported_group["positive_seeds"] == sum(value > 0 for value in values)
        assert reported_group["negative_seeds"] == sum(value < 0 for value in values)
        assert [x["seed"] for x in reported_group["paired_seed_deltas"]] == [seed for seed, _ in pairs]
        np.testing.assert_allclose([x["delta_percentage_points"] for x in reported_group["paired_seed_deltas"]], values, rtol=0, atol=1e-10)
    return {"status": "PASS", "per_seed_contrasts": len(expected), "aggregate_contrasts": len(grouped),
        "method": "Independently recomputed integer correct-prediction differences, sample SD and Student t interval; analytic two-degree-of-freedom t quantile, no runner contrast helper or SciPy used."}

def _independent_ema_interaction(plan, evaluated):
    """Compute paired architecture-by-EMA interaction within each seed first."""
    rows = []
    for seed in sorted({arm["seed"] for arm in plan["arms"]}):
        arms = [arm for arm in plan["arms"] if arm["seed"] == seed]
        score = {}
        membership = None
        for arm in arms:
            predictions = evaluated[arm["id"]]["probe"]["predictions"]
            keys = [(p["sample_id"], p["label"]) for p in predictions]
            if membership is None: membership = keys
            else: assert keys == membership
            score[(arm["kind"], arm["condition"])] = 100 * sum(p["correct_top1"] for p in predictions) / len(predictions)
        semantic_effect = score[("semantic", "ema0990")] - score[("semantic", "ema0996")]
        vit_effect = score[("vit", "ema0990")] - score[("vit", "ema0996")]
        rows.append({"seed": seed, "semantic_fast_minus_base_pp": semantic_effect,
            "vit_fast_minus_base_pp": vit_effect, "interaction_pp": semantic_effect - vit_effect})
    values = [row["interaction_pp"] for row in rows]
    assert len(values) == 3
    mean = sum(values) / len(values)
    sd = math.sqrt(sum((value - mean) ** 2 for value in values) / (len(values) - 1))
    half = math.sqrt(2 * .95**2 / (1 - .95**2)) * sd / math.sqrt(len(values))
    return {"label": "Exploratory architecture-by-EMA difference in differences",
        "formula": "(semantic fast - semantic base) - (ViT fast - ViT base), within each paired seed",
        "paired_seed_effects": rows, "n_training_seeds": 3, "mean_interaction_pp": mean,
        "sd_interaction_pp": sd, "approximate_t_ci95_pp": [mean - half, mean + half],
        "positive_means": "Faster EMA benefits semantic more than ViT in this paired experiment",
        "uncertainty": "Interactions are formed within each seed before aggregation, preserving paired covariance. Three training seeds, not development images, are the replication units. The t interval is fragile and exploratory; no multiplicity adjustment."}


def review_cpu_fixture_smoke(root):
    """Read-only CPU validation of the reconstruction on known Notebook03 evidence."""
    root = Path(root)
    artifact = root / "artifacts/notebook_03/semantic-local-pilot-002"
    split = _review_json(artifact / "split_manifest.json")["selected_ids"]
    results = {}
    torch.set_num_threads(2)
    for model_id in ("vit_trained", "semantic_trained"):
        saved = _review_json(artifact / (model_id + "_semantic_probe.json"))
        probe = saved["probe"]
        selected = torch.load(probe["summary"]["selected_probe_path"], map_location="cpu", weights_only=True)
        caches, evidence = _read_review_caches(artifact / "semantic_features" / model_id,
            split["training"], split["development"], split["heldout"], probe["summary"], selected)
        metrics = _reconstruct_probe_predictions(selected, caches, probe["predictions"], probe["summary"])
        results[model_id] = {"status": "PASS", "known_predictions_reconstructed": 2020, "metrics": metrics}
    return {"status": "PASS", "fixtures": results, "gpu_execution": False,
        "scope": "Numerical reconstruction/cache identity helpers against existing original pilot; final-epoch wrapper and full experiment audit await completed Notebook04."}

def _review_plan_binding(plan, binding):
    """Verify the explicitly documented numeric-key JSON round-trip discrepancy."""
    disk_hash = _review_hash(plan)
    reconstructed = copy.deepcopy(plan)
    changes = []
    models = reconstructed["engineering_selection"]["equivalence_certificate"]["models"]
    baseline = int(reconstructed["engineering_selection"]["equivalence_certificate"]["baseline_microbatch"])
    selected = int(reconstructed["engineering_selection"]["selected_microbatch"])
    for kind, evidence in models.items():
        losses = evidence["weighted_losses"]
        assert set(losses) == {str(value) for value in (baseline, selected)}
        assert all(key.isdigit() and str(int(key)) == key for key in losses)
        evidence["weighted_losses"] = {int(key): value for key, value in losses.items()}
        changes.append({"path": "engineering_selection.equivalence_certificate.models." + kind + ".weighted_losses",
                        "serialized_keys": list(losses), "execution_key_type": "integer microbatch"})
    # Restore only dictionary key types; every serialized value must remain identical.
    assert json.loads(json.dumps(reconstructed)) == plan
    execution_hash = _review_hash(reconstructed)
    assert binding["plan_hash"] == execution_hash, "Execution-plan identity does not reconstruct"
    return {"status": "PASS", "serialized_plan_canonical_sha256": disk_hash,
        "reconstructed_execution_plan_sha256": execution_hash,
        "roundtrip_hashes_differ": disk_hash != execution_hash, "restored_key_types": changes,
        "reason": "The original execution object used integer microbatch keys in engineering weighted_losses; JSON saves object keys as strings, whose sort order differs. No experiment parameter or recorded value changed.",
        "resume_requirement": "Restore exactly these known microbatch key types before invoking frozen runner checks in a fresh kernel; use the separate recovery notebook."}


In [ ]:
def audit_followup_run(root, run_id="semantic-followup-001"):
    root = Path(root).resolve()
    artifact = root / "artifacts/notebook_04" / run_id
    read = lambda name: _review_json(artifact / name)
    torch.set_num_threads(2)
    plan, clock, completion = read("plan.json"), read("budget_clock.json"), read("pilot_status.json")
    assert plan["run_id"] == run_id
    assert completion["status"] == "COMPLETED", "Experiment is not complete"
    allowed = float(clock["deadline_unix"]) - float(clock["started_unix"])
    assert 0 < allowed <= 10800 + 1e-6
    assert clock["started_unix"] <= completion["updated_unix"] <= clock["deadline_unix"]
    assert 0 <= completion["elapsed_seconds"] <= allowed
    assert plan["budget"]["clock"] == clock, "Original clock changed after plan registration"
    watchdog = read("deadline_watchdog_launch.json")
    assert watchdog["deadline_unix"] == clock["deadline_unix"]
    assert watchdog["worker_session"] == "fmm-followup-" + run_id
    assert digest_file(root / "experiments/04_followup/deadline_watchdog.ipynb") == watchdog["module_sha256"]
    if (artifact / "deadline_watchdog_result.json").exists():
        watch_result = read("deadline_watchdog_result.json")
        assert watch_result["status"] != "DEADLINE_ENFORCED", "Cannot approve cutoff as complete"
    assert plan["source_hashes"] == _review_source_hashes(root)
    main = _review_json(root / plan["resume_notebook"])
    consumer = _review_hash(["".join(c["source"]) for c in main["cells"] if c["cell_type"] == "code"])
    assert consumer == plan["consumer_notebook_sha256"]
    predecessor = read("predecessor_file_hashes.json")
    for name, digest in predecessor.items(): assert digest_file(root / name) == digest
    registration = read("factorial_registration.json")
    binding = registration["binding"]
    plan_binding_review = _review_plan_binding(plan, binding)
    assert binding["source_hashes"] == plan["source_hashes"]
    split_manifest = read("split_manifest.json")
    split = split_manifest["selected_ids"]
    assert len(split["training"]) == 10100 and len(split["development"]) == 2020
    assert binding["split_sha256"] == split_manifest["split_sha256"]
    assert binding["dataset_fingerprint"] == split_manifest["dataset_fingerprint"]
    assert not set(split["training"]) & set(split["development"])
    assert not (set(split["training"]) | set(split["development"])) & set(split["heldout"])
    data = read("data_reuse_evidence.json")
    assert data["verified_selected_images"] == 12120 and data["test_pixels_read"] == 0
    assert data["test_prediction_status"] == "NOT_RUN"
    preflight = read("factorial_preflight.json")
    assert preflight["status"] == "PASS" and preflight["binding"] == binding
    assert len(preflight["arms"]) == 4
    for result in preflight["arms"].values():
        assert result["bitwise_equal"] and result["status"] == "PASS"
        for entry in result["checkpoints"].values():
            path = Path(entry["directory"])
            manifest = validate_training_manifest(path)
            assert manifest["successful_updates"] == 4 and digest_file(path / manifest["latest"]) == entry["sha256"]
    training, evaluation = read("factorial_training_summary.json"), read("factorial_evaluation.json")
    assert training["status"] == evaluation["status"] == "COMPLETE"
    assert training["binding"] == evaluation["binding"] == binding
    assert training["equal_updates"] and training["paired_traces_within_seed"] and training["no_failed_attempts"]
    assert evaluation["heldout_status"] == registration["heldout_status"] == "NOT_RUN"
    arms = plan["arms"]
    assert plan["factor"] == "ema" and len(arms) == len(training["arms"]) == 12
    assert len({a["seed"] for a in arms}) == evaluation["training_seed_count"] == 3
    updates = int(plan["updates"])
    assert training["planned_updates"] == updates
    trajectory_hashes, target_hashes, initial_hashes, trainable_initials = {}, {}, {}, {}
    checkpoint_evidence = {}
    for arm in arms:
        arm_id, kind = arm["id"], arm["kind"]
        config = arm["config"]
        assert config["training"]["updates"] == updates
        effective = config["training"]["microbatch"] * config["training"]["accumulation_steps"]
        assert effective == 128
        assert not config["evaluation"]["heldout_enabled"]
        folder = root / "checkpoints" / run_id / "arms" / arm_id
        manifest = validate_training_manifest(folder)
        assert manifest["successful_updates"] == manifest["attempted_updates"] == updates
        assert manifest["image_exposures"] == manifest["cursor"] == effective * updates
        assert len(manifest["checkpoints"]) >= 2 and manifest["best"] in {x["file"] for x in manifest["checkpoints"]}
        assert manifest["resume_notebook"] == plan["resume_notebook"] and manifest["resume_cell"] == plan["resume_cell"]
        checkpoint_path = _record_verified(training["arms"][arm_id]["checkpoint"])
        assert checkpoint_path.resolve() == (folder / manifest["latest"]).resolve()
        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        assert payload["config"] == config
        protocol_path = artifact / "arms" / arm_id / "protocol.json"
        assert digest_file(protocol_path) == registration["arms"][arm_id]["protocol_sha256"]
        assert payload["protocol"] == _review_json(protocol_path)
        assert payload["scheduler"]["position"] == payload["ema_position"] == updates
        assert payload["sampler"]["cursor"] == effective * updates
        assert payload["microstep"] == 0 and not payload["partial_accumulation"]
        _finite_tensor_tree(payload["model"], arm_id + ".model")
        _finite_tensor_tree(payload["optimizer"], arm_id + ".optimizer")
        for retained_entry in manifest["checkpoints"]:
            if retained_entry["file"] == manifest["latest"]: continue
            retained = torch.load(folder / retained_entry["file"], map_location="cpu", weights_only=False)
            _finite_tensor_tree(retained["model"], arm_id + ".retained.model")
            _finite_tensor_tree(retained["optimizer"], arm_id + ".retained.optimizer")
            del retained
        state = payload["state"]
        assert state["successful_updates"] == state["attempted_updates"] == updates and not state["failures"]
        rows = state["metrics"]
        assert [row["successful_updates"] for row in rows] == list(range(1, updates + 1))
        assert all(row["teacher_gradients_absent"] and row["teacher_eval"] and row["failed_attempts_total"] == 0 for row in rows)
        assert all(np.isfinite(row["loss"]) and np.isfinite(row["gradient_norm_preclip"]) for row in rows)
        traces = [{key: row[key] for key in ("sample_ids", "occurrences", "augmentation_ids", "mask_ids")} for row in state["traces"]]
        assert len(traces) == updates and all(len(row["sample_ids"]) == effective for row in traces)
        trace_hash = _review_hash(traces)
        assert trace_hash == training["arms"][arm_id]["paired_trace_sha256"]
        trajectory_hashes.setdefault(arm["seed"], []).append(trace_hash)
        target = {name[len("target."):]: tensor for name, tensor in payload["model"].items() if name.startswith("target.")}
        target_hashes[arm_id] = encoder_state_hash(target)
        initial = registration["arms"][arm_id]["initialization"]["states"][kind]
        for record in initial.values(): _record_verified(record)
        initial_state = torch.load(initial["teacher"]["path"], map_location="cpu", weights_only=True)
        initial_hashes[arm_id] = encoder_state_hash(initial_state)
        initial_full = torch.load(initial["model"]["path"], map_location="cpu", weights_only=True)
        # State dict buffers are omega only in these registered ViT/semantic models.
        initial_trainable = {name: tensor for name, tensor in initial_full.items()
            if (name.startswith("context.") or name.startswith("predictor.")) and not name.endswith(".omega")}
        assert encoder_state_hash(initial_trainable) == registration["arms"][arm_id]["trainable_parameter_hash"]
        for name, tensor in initial_state.items():
            assert torch.equal(tensor, initial_full["context." + name])
        del initial_full, initial_trainable
        for name, tensor in payload["model"].items():
            if name.startswith("context.") and name.endswith(".omega"):
                assert torch.equal(tensor, payload["model"][name.replace("context.", "target.", 1)])
                assert torch.equal(tensor, initial_state[name[len("context."):]])
        trainable_initials.setdefault(arm["seed"], set()).add(registration["arms"][arm_id]["trainable_parameter_hash"])
        performance = training["arms"][arm_id]["performance"]
        assert performance["successful_update_coverage_complete"] and performance["history_covered_through_update"] >= updates
        assert performance["complete_jepa_step_seconds"]["median"] > 0 and performance["peak_cuda_allocated_bytes"] > 0
        checkpoint_evidence[arm_id] = {"status": "PASS", "updates": updates, "exposures": effective * updates,
            "retained_checkpoints_finite": len(manifest["checkpoints"]), "trace_sha256": trace_hash, "target_state_hash": target_hashes[arm_id], "checkpoint_sha256": digest_file(checkpoint_path)}
        del payload, target, initial_state, state, rows, traces
    assert all(len(set(values)) == 1 for values in trajectory_hashes.values())
    assert all(len(values) == 1 for values in trainable_initials.values())
    evaluated = evaluation["evaluations"]
    assert len(evaluated) == 18
    controls = evaluation["arm_initialization_controls"]
    assert set(controls) == {a["id"] for a in arms} and len(set(controls.values())) == 6
    eval_manifest = read("factorial_evaluation_manifest.json")
    assert evaluation["identity_hash"] == eval_manifest["identity_hash"]
    assert set(eval_manifest["completed"]) == set(evaluated)
    assert _review_hash(eval_manifest["identity"]) == evaluation["identity_hash"]
    reconciliation, probe_inits, probe_budgets, order_hashes = {}, set(), set(), set()
    for model_id, saved in evaluated.items():
        assert _review_json(_record_verified(eval_manifest["completed"][model_id])) == saved
        assert saved["identity_hash"] == evaluation["identity_hash"]
        _record_verified(saved["source_encoder"])
        summary = saved["probe"]["summary"]
        if saved["control"] == "trained":
            assert model_id in target_hashes and summary["checkpoint_hash"] == target_hashes[model_id]
        else:
            mapped = [arm for arm in arms if controls[arm["id"]] == model_id]
            assert mapped and saved["control"] == "initialization"
            assert all(summary["checkpoint_hash"] == initial_hashes[arm["id"]] and saved["kind"] == arm["kind"] and saved["seed"] == arm["seed"] for arm in mapped)
        reconciliation[model_id] = reconcile_final_epoch_probe(root, artifact, run_id, model_id, saved, split)
        probe_inits.add(summary["initialization_hash"])
        probe_budgets.add(_review_hash(summary["budget"]))
        order_hashes.add(reconciliation[model_id]["order_hash"])
    assert len(probe_inits) == len(probe_budgets) == len(order_hashes) == 1
    contrasts = _independent_seed_contrasts(plan, evaluated, controls, evaluation["contrasts"])
    interaction = _independent_ema_interaction(plan, evaluated)
    figures = read("factorial_figure_manifest.json")
    assert digest_file(figures["source"]) == figures["source_sha256"]
    chart = _review_json(figures["source"])
    assert len(chart) == len(arms)
    chart_by_id = {row["arm_id"]: row for row in chart}
    for arm in arms:
        row = chart_by_id[arm["id"]]
        trained = 100 * evaluated[arm["id"]]["probe"]["summary"]["metrics"]["top1"]
        initial = 100 * evaluated[controls[arm["id"]]]["probe"]["summary"]["metrics"]["top1"]
        assert math.isclose(row["trained_top1_percent"], trained, abs_tol=1e-10)
        assert math.isclose(row["initial_top1_percent"], initial, abs_tol=1e-10)
        assert math.isclose(row["gain_percentage_points"], trained - initial, abs_tol=1e-10)
    outcome = {"schema": "independent-followup-review/v1", "status": "PASS", "run_id": run_id,
        "completed_unix": completion["updated_unix"], "original_deadline_unix": clock["deadline_unix"],
        "authorized_budget_seconds": allowed, "recorded_elapsed_seconds": completion["elapsed_seconds"],
        "plan_identity_review": plan_binding_review, "source_hashes_verified": len(plan["source_hashes"]), "predecessor_files_unchanged": len(predecessor),
        "arms": checkpoint_evidence, "paired_training_seeds": 3, "updates_each": updates,
        "all_within_seed_traces_equal": True, "equal_trainable_initialization_within_seed": True,
        "initialization_controls": 6, "fixed_epoch_probes": 18, "probe_reconciliation": reconciliation,
        "shared_probe_initialization_budget_and_order": True, "contrasts": contrasts,
        "ema_architecture_interaction": interaction, "chart_values_reconciled": True, "visual_inspection": "Separate image inspection required",
        "official_test_used": False, "audit_device": "cpu",
        "review_environment": {"torch": torch.__version__, "numpy": np.__version__, "scipy": importlib.metadata.version("scipy"), "interval_quantile_method": "analytic t distribution with 2 degrees of freedom"}, "audit_finished_unix": time.time()}
    atomic_json(artifact / "independent_review.json", outcome)
    return outcome


In [ ]:
RUN_ID = "semantic-followup-001"
REVIEW = audit_followup_run(ROOT, RUN_ID)
print({key: REVIEW[key] for key in ("status", "run_id", "updates_each", "paired_training_seeds", "fixed_epoch_probes", "recorded_elapsed_seconds")})

In [ ]:
from IPython.display import display, Markdown, Image
ARTIFACT = ROOT / "artifacts/notebook_04" / RUN_ID
display(Markdown((ARTIFACT / "factorial_results.md").read_text()))
for name in ("factorial_quality.png", "factorial_compute.png"):
    path = ARTIFACT / name
    assert path.exists(), "Missing result figure: " + name
    display(Image(filename=str(path)))
interaction = REVIEW["ema_architecture_interaction"]
lines = ["### Exploratory paired EMA interaction", "", interaction["formula"], "",
    "| Training seed | Semantic fast − base | ViT fast − base | Difference in differences |",
    "|---|---:|---:|---:|"]
for row in interaction["paired_seed_effects"]:
    lines.append(f"| {row['seed']} | {row['semantic_fast_minus_base_pp']:+.2f} pp | {row['vit_fast_minus_base_pp']:+.2f} pp | {row['interaction_pp']:+.2f} pp |")
lo, hi = interaction["approximate_t_ci95_pp"]
lines += ["", f"Across three paired seeds: {interaction['mean_interaction_pp']:+.2f} pp; approximate 95% t interval [{lo:+.2f}, {hi:+.2f}] pp.", "", interaction["uncertainty"]]
display(Markdown("\n".join(lines)))
